In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agents-core/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Long-running agents — practice

Re-implement the three methods that carry the five rules. Each exercise is graded by the real tests in `tests/test_core.py`. Reference: `durable.py` itself (cover it up first).

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))                      # repo root (run from notebooks/)
from durable import Agent, Crash, FakeClock, FakeModel, LeaseHeld, PaymentAPI, Queue, Store, Wait
RUNS = "runs.json"

def fresh(script, tools=None, clock=None):
    if os.path.exists(RUNS): os.remove(RUNS)
    pay = PaymentAPI()
    return Agent(Store(RUNS), Queue(), FakeModel(script), tools or {"charge": pay}, clock=clock or FakeClock()), pay

def show(run):
    print(f"{run.id}  status={run.status}  waiting_on={run.waiting_on}  result={run.result}")
    for i, s in enumerate(run.journal):
        print(f"  [{i}] {s['type']:<8}", {k: v for k, v in s.items() if k != "type"})

import durable
sys.path.insert(0, os.path.abspath('../tests')); import test_core as T

## Exercise 1 — `Store.acquire_lease(run_id, owner, ttl, now)`  ·  rule (3)
Refuse if another owner holds an unexpired lease (raise `LeaseHeld`); otherwise set `run.lease = {"owner", "until"}` and save. Same owner may re-acquire.

In [ ]:
def acquire_lease(self, run_id, owner, ttl, now):
    run = self.get(run_id)
    # TODO
    raise NotImplementedError

durable.Store.acquire_lease = acquire_lease
T.test_two_workers_cannot_advance_the_same_run(); T.test_crash_after_side_effect_charges_once(); print("lease ✓")

## Exercise 2 — `Agent._decide(run)`  ·  rules (2) and (4)
1. If `run.decisions() >= run.max_steps` → `self._finish(run, "FAILED", "budget: ...")`, return `None`.
2. `d = self.model.decide(run.journal)`; append `{"type": "decision", **d}`.
3. If `"final" in d` → `self._finish(run, "DONE", d["final"])`, return `None`.
4. Append the intent `{"type": "intent", "tool", "args", "key": f"{run.id}:{len(run.journal)}", "done": False, "approved": False}`, **save** (checkpoint 1), call `self._crash_maybe("after_intent")`, return the intent.

In [ ]:
def _decide(self, run):
    # TODO
    raise NotImplementedError

durable.Agent._decide = _decide
T.test_happy_path_one_step_per_wakeup(); T.test_budget_stops_a_runaway_loop(); T.test_crash_after_intent_before_side_effect_executes_once(); print("decide ✓")

## Exercise 3 — `Agent._execute(run, intent)`  ·  rules (2) and (5)
1. `tool = self.tools[intent["tool"]]`; if `tool.needs_approval` and not `intent["approved"]` → `return self._park(run, intent["key"], "approval")`.
2. `result = tool(**intent["args"], key=intent["key"])`, then `self._crash_maybe("after_side_effect")`.
3. If `result` is a `Wait` → `return self._park(run, result.token, "event")`.
4. Mark the intent done with the result, **save** (checkpoint 2), `self._wake(run)`, return the run.

In [ ]:
def _execute(self, run, intent):
    # TODO
    raise NotImplementedError

durable.Agent._execute = _execute
T.test_crash_after_side_effect_charges_once(); T.test_duplicate_delivery_is_ignored(); T.test_human_gate_parks_then_executes_exactly_what_was_approved(); T.test_slow_tool_parks_until_the_world_calls_back(); print("execute ✓")

## Exercise 4 — `Agent.resume(run_id, token, payload)`  ·  rule (5)
No-op unless the run is `WAITING` and the token matches. For `"approval"`: approved → `intent["approved"] = True`; rejected → mark the intent done with `{"rejected": reason}`. For `"event"`: mark the intent done with the payload. Then `RUNNING`, clear `waiting_on`, save, `_wake`.

In [ ]:
def resume(self, run_id, token, payload):
    # TODO
    raise NotImplementedError

durable.Agent.resume = resume
T.test_human_gate_parks_then_executes_exactly_what_was_approved(); T.test_rejection_is_recorded_and_the_loop_continues(); T.test_slow_tool_parks_until_the_world_calls_back(); print("resume ✓")

## Think about it
1. Which crash window is still open after both checkpoints? (Hint: between `tool(...)` returning and `save()`.) What closes it — the key, the memo, or both?
2. Two Cloud Run instances get the same wake-up 50 ms apart. Trace both through your code.
3. Replace `FakeModel` with Gemini: what exactly is in the prompt on a retry, and why must the retry *not* call it?

In [ ]:
answers = '''
1.
2.
3.
'''